# 02 · Run experiments
Runs the full matrix **datasets × models × seeds** and writes two CSVs:
- `results/calibration.csv` — RQ1 calibration metrics
- `results/conformal_fairness.csv` — RQ2/RQ3 conformal + fairness metrics

**Start small** (the default `QUICK=True` config below runs Tier A only, 2 models, 2 seeds — minutes). Then flip to the full config for the paper.

Everything downloads automatically (folktables → US Census, OpenML → openml.org) and caches under `data/`. Nothing to place by hand.

In [ ]:
import os, pathlib, certifi
# --- Secrets: load HF_TOKEN / TABPFN_TOKEN from a local, gitignored .env if present
#     (see .env.example). Never hardcode tokens. Accept the TabPFN license once at
#     https://ux.priorlabs.ai, then put your tokens in .env.
for _p in (pathlib.Path(".env"), pathlib.Path("..") / ".env"):
    if _p.exists():
        for _line in _p.read_text().splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
        break
# macOS Python.framework needs an explicit CA bundle for TabPFN's urllib-based auth.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
os.environ.setdefault("REQUESTS_CA_BUNDLE", certifi.where())
from tabpfn import TabPFNClassifier

In [2]:
import sys, os, itertools, warnings, time
sys.path.append(os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from src.data_loaders import dataset_registry
from src.pipeline import run_cell
from src.models import MODEL_NAMES
os.makedirs('../results', exist_ok=True)

### Experiment configuration

In [3]:
QUICK = False   # full paper run; MEDIUM below scopes it to a laptop-friendly size

reg = dataset_registry()
if QUICK:
    DATASETS = ['ACSPublicCoverage-CA', 'ACSIncome-CA']
    MODELS = ['xgboost', 'lightgbm', 'mlp'] 
    SEEDS    = [0, 1]
    ALPHAS   = (0.10,)
    SCORES   = ('lac',)
else:
    # MEDIUM config: Tier-A fairness datasets (ACS) × all 5 models × 5 seeds
    # × 3 confidence levels × 2 conformity scores. Covers RQ1/RQ2/RQ3 headline
    # in well under an hour. For the full paper run add Tier-B and 10 seeds.
    tierA = [k for k, v in reg.items() if v[2] == 'A']
    DATASETS = tierA                          # ACSPublicCoverage/Income/Employment-CA
    MODELS   = MODEL_NAMES                     # tabpfn, tabpfn_temp, xgboost, lightgbm, mlp
    SEEDS    = list(range(5))
    ALPHAS   = (0.05, 0.10, 0.20)
    SCORES   = ('lac', 'aps')

print(len(DATASETS), 'datasets ×', len(MODELS), 'models ×', len(SEEDS), 'seeds')

3 datasets × 5 models × 5 seeds


### Run the matrix
Each cell is cached-friendly: results are appended and written after every cell, so you can interrupt and resume. Re-running skips nothing by default — delete the CSVs to start fresh, or add your own skip logic.

In [4]:
calib_rows, conf_rows = [], []
t0 = time.time()
for ds in DATASETS:
    loader_fn, kwargs, tier = reg[ds]
    for model in MODELS:
        for seed in SEEDS:
            try:
                cr, frs = run_cell(ds, loader_fn, kwargs, model, seed,
                                   alphas=ALPHAS, scores=SCORES)
                calib_rows.append(cr); conf_rows.extend(frs)
                print(f'[{time.time()-t0:6.1f}s] {ds:24s} {model:12s} seed={seed} '
                      f"acc={cr['accuracy']:.3f} ece={cr['ece_adaptive']:.3f}")
            except Exception as e:
                print(f'  !! FAILED {ds} {model} seed={seed}: {e}')
    # checkpoint after each dataset
    pd.DataFrame(calib_rows).to_csv('../results/calibration.csv', index=False)
    pd.DataFrame(conf_rows ).to_csv('../results/conformal_fairness.csv', index=False)
print('DONE in', round(time.time()-t0,1), 's')

[  43.3s] ACSPublicCoverage-CA     tabpfn       seed=0 acc=0.716 ece=0.040


[  87.1s] ACSPublicCoverage-CA     tabpfn       seed=1 acc=0.727 ece=0.027


[ 143.3s] ACSPublicCoverage-CA     tabpfn       seed=2 acc=0.718 ece=0.033


[ 240.4s] ACSPublicCoverage-CA     tabpfn       seed=3 acc=0.728 ece=0.022


[ 367.8s] ACSPublicCoverage-CA     tabpfn       seed=4 acc=0.711 ece=0.028


...

[3936.4s] ACSEmployment-CA         mlp          seed=0 acc=0.799 ece=0.027


[3943.7s] ACSEmployment-CA         mlp          seed=1 acc=0.795 ece=0.012


[3951.0s] ACSEmployment-CA         mlp          seed=2 acc=0.800 ece=0.021


[3958.1s] ACSEmployment-CA         mlp          seed=3 acc=0.793 ece=0.019


[3965.2s] ACSEmployment-CA         mlp          seed=4 acc=0.775 ece=0.021
DONE in 3965.2 s


In [5]:
calib = pd.DataFrame(calib_rows)
conf  = pd.DataFrame(conf_rows)
print('calibration rows:', len(calib), '| conformal rows:', len(conf))
calib.head()

calibration rows: 75 | conformal rows: 3600


,dataset,model,seed,n_classes,n_test,accuracy,ece_adaptive,ece_equalwidth,mce,brier,nll
0,ACSPublicCoverage-CA,tabpfn,0,2,2400,0.715833,0.040261,0.033394,0.057401,0.382646,0.568085
1,ACSPublicCoverage-CA,tabpfn,1,2,2400,0.726667,0.027361,0.023037,0.035519,0.367749,0.544547
2,ACSPublicCoverage-CA,tabpfn,2,2,2400,0.717917,0.033278,0.021504,0.056145,0.369329,0.546420
3,ACSPublicCoverage-CA,tabpfn,3,2,2400,0.727500,0.021802,0.019101,0.054979,0.365617,0.542739
4,ACSPublicCoverage-CA,tabpfn,4,2,2400,0.711250,0.028382,0.024810,0.062472,0.381034,0.561444


When this finishes, open **03_analysis_figures.ipynb**.